In [1]:
import joblib
import pandas as pd
import numpy as np

model = joblib.load("../models/random_forest_tuned.pkl")
scaler = joblib.load("../models/scaler.pkl")

# Reload training columns (needed to align any new input the same way)
df_train_full = pd.read_csv("../data/processed_train.csv")
train_columns = df_train_full.drop(columns=["binary_label"]).columns

print("Model, scaler, and column reference loaded.")

Model, scaler, and column reference loaded.


In [2]:
columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate"
]

def predict_intrusion(raw_row_df):
    """
    Takes a raw dataframe (one or more rows) with the original 41 feature
    columns (no label/difficulty) and returns predictions + confidence scores.
    """
    df = raw_row_df.copy()

    # Log-transform skewed features (same as Day 4)
    for col in ["duration", "src_bytes", "dst_bytes"]:
        df[col] = np.log1p(df[col])

    # One-hot encode categoricals (same as Day 4)
    df = pd.get_dummies(df, columns=["protocol_type", "service", "flag"], drop_first=True)

    # Align columns to match training exactly (same as Day 8)
    df = df.reindex(columns=train_columns, fill_value=0)

    # Scale (same as Day 5)
    df_scaled = scaler.transform(df)

    # Predict
    predictions = model.predict(df_scaled)
    probabilities = model.predict_proba(df_scaled)[:, 1]  # probability of "attack"

    results = pd.DataFrame({
        "prediction": ["attack" if p == 1 else "normal" for p in predictions],
        "confidence": probabilities
    })

    return results

In [3]:
df_test_raw = pd.read_csv("../data/KDDTest+.txt", names=columns + ["label", "difficulty"])
sample = df_test_raw.drop(columns=["label", "difficulty"]).head(5)

results = predict_intrusion(sample)
print(results)

# Compare against actual labels
print("\nActual labels:")
print(df_test_raw["label"].head(5))

  prediction  confidence
0     attack        1.00
1     attack        1.00
2     normal        0.00
3     attack        0.67
4     normal        0.35

Actual labels:
0    neptune
1    neptune
2     normal
3      saint
4      mscan
Name: label, dtype: str
